In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import brier_score
from scipy.interpolate import interp1d
from sklearn.inspection import permutation_importance
from sksurv.metrics import concordance_index_censored



In [2]:
#=== Load the data ===#

train_df = pd.read_csv("WiDSWorldWide_GlobalDathon26/train.csv")
train_df = train_df.drop("event_id", axis=1)


test_df = pd.read_csv("WiDSWorldWide_GlobalDathon26/test.csv")
test_event_id = test_df["event_id"].copy()

meta_df = pd.read_csv("WiDSWorldWide_GlobalDathon26/metaData.csv")

submission = pd.read_csv("WiDSWorldWide_GlobalDathon26/sample_submission.csv")

In [3]:
train_df.head()

,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,log_area_ratio_0_5h,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,0.03545,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,0.00000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,0.00000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,0.00000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0


In [4]:
# Check dataset dimensions
print(f"Number of training samples: {train_df.shape[0]}")
print(f"Number of features: {train_df.shape[1]}")
print(f"Number of test samples: {test_df.shape[0]}")

Number of training samples: 221
Number of features: 36
Number of test samples: 95


In [5]:
# Calculate some basic statistics
n_hits = int(train_df['event'].sum())
n_censored = len(train_df) - n_hits
hit_times = train_df.loc[train_df['event'] == 1, 'time_to_hit_hours']

# Print a summary

print(f"Training set:   {train_df.shape[0]} fires with {train_df.shape[1]} columns")
print(f"Test set:       {test_df.shape[0]} fires with {test_df.shape[1]} columns")
print()
print("Target distribution in training data:")
print(f"  Fires that HIT:        {n_hits} ({n_hits/len(train_df)*100:.1f}%)")
print(f"  Censored (no hit):     {n_censored} ({n_censored/len(train_df)*100:.1f}%)")
print()
print("Time to hit (for fires that hit):")
print(f"  Average: {hit_times.mean():.1f} hours")
print(f"  Median:  {hit_times.median():.1f} hours")
print(f"  Fastest: {hit_times.min():.2f} hours")
print(f"  Slowest: {hit_times.max():.1f} hours")
print()
print(f"Missing values: {train_df.isnull().sum().sum()}")
print(f"Unique values per column:")
print(train_df.nunique())



Training set:   221 fires with 36 columns
Test set:       95 fires with 35 columns

Target distribution in training data:
  Fires that HIT:        69 (31.2%)
  Censored (no hit):     152 (68.8%)

Time to hit (for fires that hit):
  Average: 10.0 hours
  Median:  3.5 hours
  Fastest: 0.00 hours
  Slowest: 66.9 hours

Missing values: 0
Unique values per column:
num_perimeters_0_5h              12
dt_first_last_0_5h               62
low_temporal_resolution_0_5h      2
area_first_ha                   221
area_growth_abs_0_5h             26
area_growth_rel_0_5h             26
area_growth_rate_ha_per_h        26
log1p_area_first                221
log1p_growth                     25
log_area_ratio_0_5h              26
relative_growth_0_5h             26
radial_growth_m                  26
radial_growth_rate_m_per_h       26
centroid_displacement_m          26
centroid_speed_m_per_h           26
spread_bearing_deg               26
spread_bearing_sin               26
spread_bearing_cos        

In [6]:
#=== Preprocess the data ===#


X = train_df.drop(["time_to_hit_hours", "event"], axis=1)
y = train_df[["event", "time_to_hit_hours"]]

X_test = test_df.copy()


# try this later to see if performance improves 
#one-hot encode the event_start_month column and drop the original column
X = pd.get_dummies(X, columns=["event_start_month"], drop_first=True)
#get rid of potentially redundant features
X = X.drop(["area_first_ha", "area_growth_abs_0_5h"], axis=1)


In [7]:
#get all features
feature_cols= [col for col in train_df.columns if col not in ["event","time_to_hit_hours"]]
print("Features:", feature_cols)

#caclculte correlation with target
corr_with_event = train_df[feature_cols].corrwith(train_df['event']).abs()
corr_with_time = train_df[feature_cols].corrwith(train_df['time_to_hit_hours']).abs()

print("Top features correlated with event:")
print(corr_with_event.sort_values(ascending=False).head(10) )
print()
print("Top features correlated with time to hit:")
print(corr_with_time.sort_values(ascending=False).head(10) )

Features: ['num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first', 'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h', 'radial_growth_m', 'radial_growth_rate_m_per_h', 'centroid_displacement_m', 'centroid_speed_m_per_h', 'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos', 'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h', 'dist_slope_ci_0_5h', 'closing_speed_m_per_h', 'closing_speed_abs_m_per_h', 'projected_advance_m', 'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos', 'alignment_abs', 'cross_track_component', 'along_track_speed', 'event_start_hour', 'event_start_dayofweek', 'event_start_month']
Top features correlated with event:
dist_min_ci_0_5h                0.481379
low_temporal_resolution_0_5h    0.379117
num_perimeters_0_5h             0.370501
dt_first_last_0_5h              0.352954
alignment_abs   

In [8]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    base = df[[
        "dist_min_ci_0_5h",
        "alignment_abs",
        "closing_speed_m_per_h",
        "dt_first_last_0_5h",
        "num_perimeters_0_5h",
        "spread_bearing_cos",
    ]].copy()

    base["inv_dist_min"] = 1 / (base["dist_min_ci_0_5h"] + 1e-6)
    base["threat_projection"] = base["alignment_abs"] * base["inv_dist_min"]
    base["time_to_collision_est"] = base["dist_min_ci_0_5h"] / (base["closing_speed_m_per_h"] + 1e-6)
    base["directed_growth"] = base["time_to_collision_est"] * base["threat_projection"]
    base["signal_confidence"] = (
        base["num_perimeters_0_5h"] * base["dt_first_last_0_5h"] *
        (1 - (base["spread_bearing_cos"] > 0.5).astype(int))
    )

    return base

In [9]:
#== Prepare data for modeling ===#

X = make_features(train_df)
X_test = make_features(test_df)

y = np.array(
    list(zip(train_df["event"].astype(bool), train_df["time_to_hit_hours"].astype(float))),
    dtype=[("event", bool), ("time", float)]
)


# Split the data into training and validation sets
X_train, X_val, y_train_event, y_val_event, y_train_time, y_val_time = train_test_split(
    X, train_df['event'], train_df['time_to_hit_hours'], test_size=0.2, random_state=42
)

In [10]:

# Create structured array for survival data
y_train = np.array(list(zip(y_train_event, y_train_time)),
                   dtype=[('event', bool), ('time', float)])
y_val = np.array(list(zip(y_val_event, y_val_time)),
                 dtype=[('event', bool), ('time', float)])

rsf = RandomSurvivalForest(
    n_estimators=1200,
    max_depth=None,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features="log2",
    n_jobs=-1,
    random_state=42
)

rsf.fit(X_train, y_train)

RandomSurvivalForest(max_features='log2', min_samples_leaf=10,
                     min_samples_split=20, n_estimators=1200, n_jobs=-1,
                     random_state=42)

In [11]:
#== Permutation importance ===#
result = permutation_importance(
    rsf,
    X_train,
    y_train,
    n_repeats=10,
    random_state=42,
)

importances = result.importances_mean

for f, imp in sorted(zip(X_train.columns, importances),
                     key=lambda x: x[1],
                     reverse=True)[:10]:
    print(f"{f}: {imp:.4f}")

dist_min_ci_0_5h: 0.0176
inv_dist_min: 0.0132
time_to_collision_est: 0.0037
num_perimeters_0_5h: 0.0030
directed_growth: 0.0020
dt_first_last_0_5h: 0.0014
threat_projection: 0.0014
alignment_abs: 0.0007
spread_bearing_cos: 0.0004
signal_confidence: 0.0001


In [12]:
surv_probs_test = rsf.predict_survival_function(X_test, return_array=True)

#Horizons
times = np.array([12, 24, 48, 72])
time_indices = []
for t in times:
    idx = np.searchsorted(rsf.unique_times_, t, side="right") - 1
    idx = np.clip(idx, 0, len(rsf.unique_times_) - 1)
    time_indices.append(idx)

probs_test = {t: 1 - surv_probs_test[:, idx] for t, idx in zip(times, time_indices)}

probs_12h = probs_test[12.0]
probs_24h = probs_test[24.0]
probs_48h = probs_test[48.0]
probs_72h = probs_test[72.0]

# Verify lengths
print(len(probs_12h), len(test_df))

95 95


In [13]:
#== Evaluate model ===#

c_index = concordance_index_censored(y_val['event'], y_val['time'], rsf.predict(X_val))
print("C-index:", c_index[0])

C-index: 0.9194630872483222


In [14]:
risk_scores = rsf.predict(X_val)

c_index = concordance_index_censored(
    y_val["event"],
    y_val["time"],
    risk_scores
)[0]

print("Validation C-index:", c_index)

Validation C-index: 0.9194630872483222


In [15]:
# Competition horizons
times_eval = np.array([24.0, 48.0, 72.0])

# Ensure horizons are valid for validation fold
max_time = y_val["time"].max()
times_eval = times_eval[times_eval < max_time]

print("Using horizons:", times_eval)

# Predict survival probabilities
surv_probs_val = rsf.predict_survival_function(X_val, return_array=True)

# Find index of closest survival time for each horizon
time_indices = [
    np.searchsorted(rsf.unique_times_, t, side="right") - 1
    for t in times_eval
]

# Extract survival probabilities at each horizon
surv_at_times = np.vstack([
    surv_probs_val[:, idx] for idx in time_indices
]).T

# Compute Brier scores (uses IPCW automatically)
_, brier_scores = brier_score(
    y_train,   # MUST use training data here for censoring distribution
    y_val,
    surv_at_times,
    times_eval
)

# Print individual Brier scores
for t, score in zip(times_eval, brier_scores):
    print(f"Brier @{t}h:", score)

# ===== WEIGHTED BRIER

weights = {24.0: 0.3, 48.0: 0.4, 72.0: 0.3}
weighted_brier = float(np.sum([
    weights[t] * score
    for t, score in zip(times_eval, brier_scores)
]))



print("Type c_index:", type(c_index))
print("Type weighted_brier:", type(weighted_brier))

# ===== FINAL HYBRID SCORE
c_index = float(c_index)
weighted_brier = float(weighted_brier)

hybrid_score = 0.3 * c_index + 0.7 * (1 - weighted_brier)

print("Hybrid Score:", hybrid_score)

Using horizons: [24. 48.]
Brier @24.0h: 0.04709910975068918
Brier @48.0h: 0.029851655875237037
Type c_index: <class 'numpy.float64'>
Type weighted_brier: <class 'float'>
Hybrid Score: 0.9575896494817855


In [16]:
print("C-index:", c_index)
print("Weighted Brier:", weighted_brier)

C-index: 0.9194630872483222
Weighted Brier: 0.02607039527530157


In [17]:
#== Prepare submission ===#

sub = pd.DataFrame({
    "event_id": test_event_id.astype(np.int64),
    "prob_12h": probs_12h,
    "prob_24h": probs_24h,
    "prob_48h": probs_48h,
    "prob_72h": probs_72h,
})

In [18]:
sub.to_csv("submission.csv", index=False)
print("Saved: submission.csv")
print(sub.head())

Saved: submission.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.006336  0.010335  0.011565  0.027286
1  13353600  0.525081  0.823401  0.890070  0.959645
2  13942327  0.006336  0.010335  0.011565  0.027286
3  16112781  0.514586  0.813426  0.877402  0.965068
4  17132808  0.229454  0.264358  0.275691  0.289346


In [19]:
print("Train C-index:", 
      concordance_index_censored(y_train["event"], y_train["time"], 
                                 rsf.predict(X_train))[0])
print("Val C-index:", c_index)

Train C-index: 0.953012215435869
Val C-index: 0.9194630872483222
